# Day 2.6 — Evaluate Retrieval Separately from Answers

If an answer is wrong, first ask whether the right evidence was retrieved. A **golden set** stores questions and expected behaviour known in advance.

## Before you begin

### Learning outcomes

Calculate retrieval success from a golden set and compare top-k settings.

Architecture reference: [D07](../../diagrams/source/day_02.md).

### Expected observation

Changing top-k changes section recall and may add irrelevant context.


## Concept briefing

## Diagnosing a bad answer

Use evidence in this order:

1. What exactly was the query?
2. Which chunks were retrieved and with what scores?
3. Does any retrieved chunk contain sufficient evidence?
4. Which chunk should have appeared according to the golden set?
5. If good evidence was present, did generation use it?
6. Did citation validation accept a source that was not actually retrieved?

If the correct evidence is absent, investigate ingestion, chunking, representation and
retrieval. If it is present but the answer is wrong, investigate context construction,
instructions, generation and validation. This separation prevents endless prompt edits
when the retriever never supplied the answer.


In [ ]:
import os,sys
from pathlib import Path
here=Path.cwd().resolve(); candidates=[here,here/"day_02_knowledge_and_state",here.parent]
project_root=next(p for p in candidates if (p/"src"/"knowledge_agent").exists())
sys.path.insert(0,str(project_root/"src"))
from knowledge_agent.documents import load_markdown_corpus
from knowledge_agent.embeddings import SentenceTransformerEmbedder
from knowledge_agent.evaluation import evaluate_retrieval,load_golden_set,summarize
from knowledge_agent.retrieval import VectorIndex
cases=load_golden_set(project_root/"data"/"golden_set.json")
chunks=load_markdown_corpus(project_root/"data"/"corpus")
index=VectorIndex(SentenceTransformerEmbedder(os.getenv("EMBEDDING_MODEL","sentence-transformers/all-MiniLM-L6-v2")))
index.add(chunks)

## Inspect the evaluation contract

The expected source and section evaluate retrieval. Essential terms and answerability are used later for answer evaluation. The golden file is not indexed or shown to the model.

In [ ]:
for case in cases[:3]: print(case.model_dump())

In [ ]:
records=evaluate_retrieval(index,cases,top_k=3)
for record in records:
    print(record["id"],"source=",record["source_hit"],"section=",record["section_hit"],record["retrieved_sections"])
print(summarize(records,["source_hit","section_hit"]))

## Compare top-k

Increasing top-k may improve recall but adds irrelevant context, tokens, and opportunities for distraction.

In [ ]:
for k in [1,2,3,5]:
    report=evaluate_retrieval(index,cases,top_k=k)
    answerable=[r for r in report if r["answerable"]]
    hit=sum(r["section_hit"] for r in answerable)/len(answerable)
    print("top_k=",k,"exact-section hit rate=",round(hit,2))

## Exercise and checkpoint

Choose one failed case, inspect its query and retrieved chunks, and propose one change to chunking, metadata, query wording, embeddings, or top-k. Change one factor and rerun the same golden set. Evaluation is the instrument for improvement, not a decorative final score.

## Your turn

Hand-calculate one case before checking the helper result.

## Recap

Evaluation converts retrieval tuning from guesswork into measurement.
